# [SQL 재현] 2023년 의료기관별 시군구별 진료비 분석

## 단계: 02. 탐색적 데이터 분석(EDA) — SQL 재현
- 목표: PY_02에서 pandas로 만든 파생변수·상위 지역·시도별 집계를 SQL로 다시 작성하고, 결과가 PY_02와 같은지 대조한다.
- 환경: Jupyter Notebook + sqlite3 (SQL_01에서 만든 sql_practice.db의 hira 테이블 사용)
- 대조 기준: PY_02_EDA.ipynb 실행 결과
- 범위 밖: 시각화·상관계수·왜도(Python 분석 영역), 로그 변환(SQL_03에서 확인)

### 2.1 환경 설정
#### 2.1-1 DB 연결 및 저장된 테이블 확인
- SQL_01에서 만든 sql_practice.db에 다시 연결한다. 테이블이 DB 파일에 저장돼 있으므로 CSV를 다시 읽을 필요가 없다.
- `sqlite_master`: SQLite가 DB 안의 테이블 목록을 기록해 두는 시스템 표

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\data\sql_practice.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,hira


### 2.2 파생변수 생성
#### 2.2-1 파생변수를 포함한 분석용 테이블 만들기
- PY_02 2.1 파생변수 대응 (SAS `DATA hira.analysis; SET hira.raw;` 대응)
- `CREATE TABLE 새테이블 AS SELECT ...`: 조회 결과를 새 테이블로 저장
- 정수 열끼리 나눌 때는 `CAST(열 AS REAL)`로 소수형으로 바꾼 뒤 나눈다 (SQL_01 1.4-1 결과 반영)
- 숫자로 시작하는 이름은 SQL에서 매번 따옴표가 필요하므로, SAS와 같이 `일인당_`으로 시작